# Week 6 Lab — Lists & Strings in the Real World

**192-101 Computer Programming with Generative AI**

Reference: *Think Python 3e*, Ch. 8 (Strings) + Ch. 9 (Lists).

---

### Why this lab matters
Almost every real system receives **messy text** — exported sales data, passwords typed by users, ID numbers with dashes, hashtags with random capitals — and has to **clean, check, and summarise** it. Today you will write the kind of small functions that real developers write every week.

| Part | Task | Mode |
|---|---|---|
| 0 | Predict the output (6 traces) | pen first |
| 1 | Coffee-shop sales report from raw POS text | **together** |
| 2 | Password strength checker | **by yourself** | 35 min |
| 3 | Thai National ID validator (checksum) | **by yourself** |
| 4 | Debug the AI: fix a broken function | **by yourself** |
| 5 | *Stretch:* student username generator | optional |

### Rules
- **Only use what we have learned:** strings, lists, loops, `if`, functions. **No dictionaries** (that's Week 7) and no `import`.
- Every activity has a **test cell**. Your work is done when every line shows ✅.
- **Do not change the test cells.** Change your function until the tests pass.

> **AI use — Level 1 (explain only):** you may ask an assistant to *explain* a concept or an error message, but you type all code yourself. Do the Part 0 traces with a pen first.

### Setup — run this cell first
It defines a small `check()` helper used by all the test cells.

In [60]:
passed = 0
failed = 0

def check(label, got, expected):
    '''Print ✅ if got == expected, otherwise ❌ with details.'''
    global passed, failed
    if got == expected:
        passed = passed + 1
        print('✅', label)
    else:
        failed = failed + 1
        print('❌', label)
        print('     expected:', repr(expected))
        print('     got     :', repr(got))

print('Setup complete.')

Setup complete.


## Part 0 — Warm-up: predict the output

Write your prediction on paper **before** running each cell.

**Trace A** — slicing with a step and negative indices.

In [61]:
s = 'Siam University'
print(s[5:9])
print(s[-3:])
print(s[::2])
print(s[::-1][:4])

Univ
ity
Sa nvriy
ytis


**Trace B** — `split` and `strip` (the #1 tools for cleaning real data).

In [62]:
line = ' apple, banana ,,cherry '
parts = line.split(',')
print(len(parts))
print(parts[1])
print('[' + parts[1].strip() + ']')
print(parts[2] == '')

4
 banana 
[banana]
True


**Trace C** — aliasing vs. copying. *(Very common exam question.)*

In [63]:
a = [1, 2, 3]
b = a
c = a[:]
b.append(4)
c.append(5)
print(a)
print(b)
print(c)
print(a is b, a is c)

[1, 2, 3, 4]
[1, 2, 3, 4]
[1, 2, 3, 5]
True False


**Trace D** — methods that change a list return `None`.

In [64]:
nums = [3, 1, 2]
result = nums.sort()
print(result)
print(nums)
print(sorted('cab'))

None
[1, 2, 3]
['a', 'b', 'c']


**Trace E** — strings are immutable.

In [65]:
s = 'hello'
t = s.upper()
s.replace('l', 'L')
print(s)
print(t)
s = s.replace('l', 'L')
print(s)

hello
HELLO
heLLo


**Trace F** — a loop over indices, plus `find` and `count`.

In [66]:
word = 'banana'
total = 0
for i in range(len(word)):
    if word[i] == 'a' and i % 2 == 1:
        total = total + i
print(total)
print(word.find('n'), word.find('z'))
print(word.count('an'))

9
2 -1
2


> **Check yourself:** for every trace you got wrong, write one sentence explaining *why*. Most mistakes come from: forgetting the stop index is excluded, forgetting that spaces are characters, aliasing, or expecting a string method to change the original.

## Part 1 — Activity 1: Coffee-Shop Sales Report  *(we build this together)*

**Real-world situation.** A small café near campus exports its daily sales from the POS (point-of-sale) machine as plain text. The data is **messy**: extra spaces, inconsistent capital letters, blank lines, and a header row. The owner wants a clean summary.

**Input** (already stored in the variable `raw_sales` below):
```
date,item,price,qty
2026-09-01, Iced Latte ,65,2
...
```

**Target output:**
```
===== DAILY SALES REPORT =====
Item              Qty    Revenue
--------------------------------
Iced Latte          5     325.00
Croissant           6     270.00
Thai Tea            7     315.00
Americano           4     200.00
--------------------------------
TOTAL              22   1,110.00
Best seller by revenue: Iced Latte
```

We will build it in **three small functions** (this is how professionals break a problem down):

1. `parse_line(line)` → turn one text line into a list `[item, price, qty]`
2. `summarise(raw_text)` → return three **parallel lists**: `names`, `qtys`, `revenues` (one entry per *unique* item, in the order first seen)
3. `print_report(names, qtys, revenues)` → print the formatted table

In [67]:
# The raw export from the POS machine (do not edit)
raw_sales = '''date,item,price,qty
2026-09-01, Iced Latte ,65,2
2026-09-01,croissant,45,3
2026-09-01,THAI TEA,45,4

2026-09-01,Americano ,50,1
2026-09-01, iced latte,65,3
2026-09-01,Croissant,45,3
2026-09-01,thai tea ,45,3
2026-09-01,americano,50,3
'''

### Step 1 — `parse_line`
Given `'2026-09-01, Iced Latte ,65,2'` return `['Iced Latte', 65, 2]`.

Plan: `split(',')` → `strip()` each piece → make the item name `.title()` → convert price and qty with `int()`.

In [68]:
def parse_line(line):
    '''Return [item, price, qty] from one CSV line.'''
    # TODO (together):
    #   1. split the line on ','
    #   2. item  = the 2nd piece, stripped, in Title Case
    #   3. price = the 3rd piece, stripped, converted to int
    #   4. qty   = the 4th piece, stripped, converted to int
    #   5. return [item, price, qty]
    parts = line.split(',')
    item = parts[1].strip().title()
    price = int(parts[2].strip())
    qty = int(parts[3].strip())
    return [item, price, qty]


# quick test
check('parse_line basic', parse_line('2026-09-01, Iced Latte ,65,2'), ['Iced Latte', 65, 2])
check('parse_line messy case', parse_line('2026-09-01,THAI TEA,45,4'), ['Thai Tea', 45, 4])

✅ parse_line basic
✅ parse_line messy case


### Step 2 — `summarise`
Loop over every line of the text and combine items with the same name.

Because we don't have dictionaries yet, we use **parallel lists**: `names[i]`, `qtys[i]` and `revenues[i]` all describe the **same** item.

Key ideas:
- `raw_text.split('\n')` gives a list of lines.
- Skip the **header** line and any **blank** line (`line.strip() == ''`).
- If the item is already in `names`, find its position with `names.index(item)` and **add** to that position. Otherwise **append** a new entry to all three lists.

In [69]:
def summarise(raw_text):
    '''Return [names, qtys, revenues] as three parallel lists.'''
    names = []
    qtys = []
    revenues = []
    lines = raw_text.split('\n')
    # TODO (together):
    #   for each line after the header (hint: lines[1:])
    #       skip it if it is blank
    #       item, price, qty = parse_line(line)
    #       if item is already in names -> find index, add qty and price*qty
    #       else -> append item, qty, price*qty
    for line in lines[1:]:
        if line.strip() == '':
            continue
        item, price, qty = parse_line(line)
        if item in names:
            i = names.index(item)
            qtys[i] += qty
            revenues[i] += price * qty
        else:
            names.append(item)
            qtys.append(qty)
            revenues.append(price * qty)
    return [names, qtys, revenues]


result = summarise(raw_sales)
check('names',    result[0], ['Iced Latte', 'Croissant', 'Thai Tea', 'Americano'])
check('qtys',     result[1], [5, 6, 7, 4])
check('revenues', result[2], [325, 270, 315, 200])

✅ names
✅ qtys
✅ revenues


### Step 3 — `print_report`
Use **f-string format specifiers** to line up the columns:
- `f'{name:<15}'` → left-align in 15 characters
- `f'{qty:>6}'` → right-align in 6 characters
- `f'{money:>11,.2f}'` → right-align, comma separators, 2 decimal places

The best seller is the name at the **same index** as `max(revenues)`.

In [70]:
def print_report(names, qtys, revenues):
    print('===== DAILY SALES REPORT =====')
    print(f'{"Item":<15}{"Qty":>6}{"Revenue":>11}')
    print('-' * 32)
    # TODO (together):
    #   loop with an index i over range(len(names)) and print one row per item
    #   print the dashed line again
    #   print the TOTAL row using sum(qtys) and sum(revenues)
    #   find the best seller: names[revenues.index(max(revenues))]

    for i in range(len(names)):
        print(f'{names[i]:<15}{qtys[i]:>6}{revenues[i]:>11,.2f}')

    print('-' * 32)
    print(f'{"Total":<15}{sum(qtys):>6}{sum(revenues):>11,.2f}')
    print(f'Best seller by revenue: {names[revenues.index(max(revenues))]}')

# This call must be OUTSIDE the function (no indentation)
print_report(result[0], result[1], result[2])


===== DAILY SALES REPORT =====
Item              Qty    Revenue
--------------------------------
Iced Latte          5     325.00
Croissant           6     270.00
Thai Tea            7     315.00
Americano           4     200.00
--------------------------------
Total              22   1,110.00
Best seller by revenue: Iced Latte


> **Discuss:** the owner now wants the report **sorted by revenue, highest first**. Why can't we simply call `revenues.sort()`? *(Hint: what happens to the connection between `names[i]` and `revenues[i]`?)*

## Part 2 — Activity 2: Password Strength Checker  *(solve by yourself)*

**Real-world situation.** You are building the sign-up page for a student portal. Before a new password is saved, the system must explain **every** problem so the user can fix it.

Write `check_password(password, username)` that returns a **list of problem messages** (an empty list means the password is fine). Check the rules **in this order** and use these **exact** messages:

| # | Rule | Message to append |
|---|---|---|
| 1 | length must be **at least 8** | `'too short'` |
| 2 | must contain an **uppercase** letter | `'no uppercase'` |
| 3 | must contain a **lowercase** letter | `'no lowercase'` |
| 4 | must contain a **digit** | `'no digit'` |
| 5 | must contain a **special** character from `SPECIALS` | `'no special character'` |
| 6 | must **not** contain a space | `'contains space'` |
| 7 | must **not** contain the username (ignore upper/lower case) | `'contains username'` |
| 8 | must **not** be in `COMMON_PASSWORDS` (ignore upper/lower case) | `'too common'` |

Then write `strength(password, username)` that returns:
- `'Strong'` if there are **0** problems
- `'Medium'` if there are **1 or 2** problems
- `'Weak'` if there are **3 or more** problems

*Hints:*
- Loop over the characters: `for ch in password:` and use `ch.isupper()`, `ch.islower()`, `ch.isdigit()`, `ch in SPECIALS`.
- Use a variable like `has_upper = False` and set it to `True` when you find one.
- For "ignore case", compare `password.lower()` with `username.lower()`.
- `strength` should **call** `check_password` — don't repeat the rules.

In [71]:
SPECIALS = '!@#$%^&*-_+=?'
COMMON_PASSWORDS = ['password', '12345678', 'qwerty123', 'admin123',
                    'iloveyou', 'password1', 'welcome1', 'abc12345']


def check_password(password, username):
    '''Return a list of problems found in password (empty list = OK).'''
    problems = []
    # Write your solution here
    # (remember to return problems at the end)
    has_upper = False
    has_lower = False
    has_digit = False
    has_special = False
    for ch in password:
        if ch.isupper():
            has_upper = True
        if ch.islower():
            has_lower = True
        if ch.isdigit():
            has_digit = True
        if ch in SPECIALS:
            has_special = True
    if len(password) < 8:
        problems.append('too short')
    if has_upper == False:
        problems.append('no uppercase')
    if has_lower == False:
        problems.append('no lowercase')
    if has_digit == False:
        problems.append('no digit')
    if has_special == False:
        problems.append('no special character')
    if ' ' in password:
        problems.append('contains space')
    if username.lower() in password.lower():
        problems.append('contains username')
    if password.lower() in COMMON_PASSWORDS:
        problems.append('too common')
    return problems


def strength(password, username):
    '''Return 'Strong', 'Medium' or 'Weak'.'''
    # Write your solution here
    problems = check_password(password, username)
    if len(problems) == 0:
        return 'Strong'
    elif len(problems) <= 2:
        return 'Medium'
    else:
        return 'Weak'

**Tests — run after writing your functions. Do not edit.**

In [72]:
check('strong password',   check_password('Siam#2026ok', 'somchai'), [])
check('too short',         check_password('Ab1!', 'somchai'), ['too short'])
check('only lowercase',    check_password('coffeeshop', 'somchai'),
      ['no uppercase', 'no digit', 'no special character'])
check('has space',         check_password('My Pass#99', 'somchai'), ['contains space'])
check('username inside',   check_password('Somchai@2026', 'somchai'), ['contains username'])
check('common (any case)', check_password('PASSWORD1', 'nok'),
      ['no lowercase', 'no special character', 'too common'])
check('empty password',    check_password('', 'nok'),
      ['too short', 'no uppercase', 'no lowercase', 'no digit', 'no special character'])

check('strength Strong',   strength('Siam#2026ok', 'somchai'), 'Strong')
check('strength Medium',   strength('Somchai@2026', 'somchai'), 'Medium')
check('strength Weak',     strength('coffeeshop', 'somchai'), 'Weak')

✅ strong password
✅ too short
✅ only lowercase
✅ has space
✅ username inside
✅ common (any case)
✅ empty password
✅ strength Strong
✅ strength Medium
✅ strength Weak


**Try it yourself (optional):** run the cell below and test your own passwords. Never type a password you really use!

In [73]:
# pw = input('New password: ')
# print(strength(pw, 'student'))
# for p in check_password(pw, 'student'):
#     print(' -', p)
pw = 'OkBroNarSr@12'
print(strength(pw, 'okbrobrlot@12'))
for p in check_password(pw, 'okbrobrlot@12'):
    print(' -', p)

Strong


## Part 3 — Activity 3: Thai National ID Validator  *(solve by yourself)*

**Real-world situation.** Banks, hospitals and university registration systems must reject mistyped ID numbers **before** saving them. The 13th digit of a Thai ID card is a **check digit** calculated from the first 12, so most typing mistakes can be detected instantly.

> 🔒 **Privacy rule:** never use a real person's ID number in this lab (not even your own). Use only the fake numbers in the tests.

**The checksum algorithm**
1. Take the first 12 digits d₀ … d₁₁.
2. Multiply d₀ by 13, d₁ by 12, d₂ by 11, … d₁₁ by 2, and add them all up → `total`.
3. `check = (11 - total % 11) % 10`
4. The ID is valid if `check` equals the 13th digit d₁₂.

*Worked example* for `1101700230708`:
```
digits : 1  1  0  1  7  0  0  2  3  0  7  0  | 8
weights: 13 12 11 10 9  8  7  6  5  4  3  2
total  = 13+12+0+10+63+0+0+12+15+0+21+0 = 146
146 % 11 = 3  →  (11 - 3) % 10 = 8  ✔ matches the last digit
```

Write **three** functions:

1. `clean_id(raw)` — people type IDs as `'1-1017-00230-70-8'` or `' 1101700230708 '`. Remove all **dashes** and **spaces** and return the digits string.
2. `is_valid_thai_id(raw)` — clean the input, then return `True` only if it is **exactly 13 characters**, **all digits**, the first digit is **not `'0'`**, and the **checksum** matches. Otherwise `False`.
3. `format_id(raw)` — return the cleaned ID in the official display format `'X-XXXX-XXXXX-XX-X'` using **slicing**. (You may assume the input is valid.)

*Hints:* `replace('-', '')`, `isdigit()`, `int(digits[i])`, weight = `13 - i`.

In [74]:
def clean_id(raw):
    # Write your solution here
    raw = raw.replace('-', '')
    raw = raw.replace(' ', '')
    return raw


def is_valid_thai_id(raw):
    # Write your solution here
    cleaned = clean_id(raw)
    if len(cleaned) != 13:
        return False
    for char in cleaned:
        if not char.isdigit():
            return False
    if cleaned[0] == '0':
        return False
    total = 0
    for i in range(12):
        total += int(cleaned[i]) * (13 - i)

    check = (11 - total % 11) % 10
    if check != int(cleaned[12]):
        return False
    return True


def format_id(raw):
    # Write your solution here
    cleaned = clean_id(raw)
    return cleaned[0] + '-' + cleaned[1:5] + '-' + cleaned[5:10] + '-' + cleaned[10:12] + '-' + cleaned[12]

**Tests — do not edit.**

In [75]:
check('clean dashes',      clean_id('1-1017-00230-70-8'), '1101700230708')
check('clean spaces',      clean_id(' 3100 5001 2345 8 '), '3100500123458')

check('valid #1',          is_valid_thai_id('1101700230708'), True)
check('valid #2 (dashes)', is_valid_thai_id('3-1005-00123-45-8'), True)
check('valid #3 (spaces)', is_valid_thai_id(' 5209900888773 '), True)
check('wrong check digit', is_valid_thai_id('1101700230709'), False)
check('two digits swapped',is_valid_thai_id('1107100230708'), False)
check('too short',         is_valid_thai_id('110170023070'), False)
check('contains letter',   is_valid_thai_id('11017002307O8'), False)
check('starts with 0',     is_valid_thai_id('0101700230708'), False)
check('empty string',      is_valid_thai_id(''), False)

check('format',            format_id('1101700230708'), '1-1017-00230-70-8')
check('format messy',      format_id(' 3100 5001 2345 8'), '3-1005-00123-45-8')

✅ clean dashes
✅ clean spaces
✅ valid #1
✅ valid #2 (dashes)
✅ valid #3 (spaces)
✅ wrong check digit
✅ two digits swapped
✅ too short
✅ contains letter
✅ starts with 0
✅ empty string
✅ format
✅ format messy


> **Think about it:** the test `'two digits swapped'` fails the checksum. Why do real systems use a check digit instead of just checking the length? Write one sentence.

## Part 4 — Debug the AI  *(solve by yourself)*

**Real-world situation.** A marketing team collects hashtags from social-media posts. A colleague asked an AI assistant to write `clean_tags` and pasted the result below **without testing it**. It looks reasonable… but it is wrong.

**The specification:**
`clean_tags(tags)` takes a list of raw tags and returns a **new** list where each tag is:
- stripped of spaces, **lowercase**, and without any `'#'` characters;
- **empty tags are removed**;
- **duplicates are removed**, keeping the **first-seen order**;
- and the **original list must not be changed**.

**Your job:**
1. Run the tests first and read which ones fail.
2. Find and fix **all the bugs** (there are **4**). Add a short comment `# FIX:` next to each line you change.
3. In the markdown cell underneath, explain each bug in one sentence.

In [76]:
# ⚠️ AI-generated code — contains bugs
def clean_tags(tags):
    result = []
    for i in range(len(tags)):
        t = tags[i]
        t = t.strip().lower()
        t = t.replace('#', '')
        if t and t not in result:
            result.append(t)

    return result


**Tests — do not edit.**

In [77]:
raw_tags = ['#SiamU', ' python ', '#Python', '', 'siamu', '# ', 'AI', '#ai']
original = raw_tags[:]           # a copy, to check the input is not changed

cleaned = clean_tags(raw_tags)
check('cleaned tags',          cleaned, ['siamu', 'python', 'ai'])
check('input list unchanged',  raw_tags, original)
check('empty input',           clean_tags([]), [])

✅ cleaned tags
✅ input list unchanged
✅ empty input


**Explain the bugs** (double-click to edit):

1. …
2. …
3. …
4. …

## Part 5 — Stretch (optional): Student Username Generator

**Real-world situation.** IT services must create e-mail usernames for new students from a registration list. Names are typed inconsistently.

Write `make_usernames(full_names)` that returns a list of usernames, one per name, in the same order:
- Clean the name: remove extra spaces (a name may have several spaces between words), make it lowercase.
- Username = **first name** + `'.'` + **first 2 letters of the last name** (the last word).
- If that username is **already taken** by an earlier student, add `2`, then `3`, … until it is unique.

| Input | Username |
|---|---|
| `'  Somchai   Jaidee '` | `'somchai.ja'` |
| `'SOMCHAI JAMES'` | `'somchai.ja2'` |
| `'somchai  jantra'` | `'somchai.ja3'` |

*Hint:* `'  a   b  '.split()` (with **no** argument) splits on any amount of whitespace and ignores the ends → `['a', 'b']`.

In [78]:
def make_usernames(full_names):
    # Write your solution here
    usernames = []
    for full_name in full_names:
        full_name = full_name.strip().lower()
        name_parts = full_name.split()
        first_name = name_parts[0]
        last_name = name_parts[-1]
        username = first_name + '.' + last_name[0:2]

        i = 2
        while username in usernames:
            username = first_name + '.' + last_name[0:2] + str(i)
            i += 1

        usernames.append(username)
    return usernames

In [79]:
names = ['  Somchai   Jaidee ', 'SOMCHAI JAMES', 'Nok Wong', 'somchai  jantra', 'Anan Lee', 'anan  LEK']
check('usernames', make_usernames(names),
      ['somchai.ja', 'somchai.ja2', 'nok.wo', 'somchai.ja3', 'anan.le', 'anan.le2'])
check('middle name ignored', make_usernames(['Mary Jane Watson']), ['mary.wa'])

✅ usernames
✅ middle name ignored


## Final score — run this last

In [80]:
print(f'Passed: {passed}   Failed: {failed}')
if failed == 0:
    print('🎉 All tests passed. Well done!')
else:
    print('Keep going - scroll up and fix the ❌ lines. (Re-run from the top: Runtime → Run all.)')

Passed: 33   Failed: 0
🎉 All tests passed. Well done!


## Before you finish — checklist

- [ ] I can explain every Part 0 trace, especially **aliasing** (Trace C) and **immutability** (Trace E).
- [ ] Activity 1: I understand how **parallel lists** keep names, quantities and revenues connected.
- [ ] Activity 2: all password tests pass, and I know why each flag needs its own `if`.
- [ ] Activity 3: all ID tests pass, and I check length/digits **before** the checksum loop.
- [ ] Activity 4: I found all 4 AI bugs and explained them in my own words.
- [ ] I did **not** use dictionaries, and I typed all code myself (AI Level 1).

**Before Week 7:** look back at Activity 1. Using `names.index(item)` to find an item gets slow and awkward with many items. Next week, **dictionaries** will let us write `sales['Iced Latte']` directly.